In [1]:
!pip install leafmap

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import plotly.express as px
import geopandas as gpd
import pandas as pd 
import nbformat
from leafmap import maplibregl
import leafmap

In [4]:
ispi_standoffs = pd.read_excel("data/ISPI - Standoffs in the Central Mediterranean (Crisi in mare).xlsx", header=1)

In [5]:
ispi_standoffs

,crisis number,ship,persons onboard,start date,end date,length of standoff,relocated (agreed),disembarked,where disembarked (Italy),nautical miles from Lampedusa
0,1,Aquarius,629,2018-09-06 00:00:00,17/06/2018,9,0,Spagna,NaN,NaN
1,2,Diciotti,523,15/06/2018,19/06/2018,5,0,Italia,Pozzallo,130.0
2,3,Lifeline,234,21/06/2018,28/06/2018,8,220,Malta,NaN,NaN
3,4,Alexander Maersk,113,22/06/2018,26/06/2018,5,0,Italia,Pozzallo,130.0
4,5,Monte Sperone + Protector,447,13/07/2018,16/07/2018,4,270,Italia,Pozzallo,130.0
...,...,...,...,...,...,...,...,...,...,...
290,Governo Draghi,79,3.891626,5.936709,6,23.103448,185,186,NaN,NaN
291,Governo Meloni,139,10.220588,3.560284,4,36.544118,410,69,NaN,NaN
292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
293,Dati liberamente utilizzabili. Autori: Matteo ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
ispi_standoffs['disembarked'].unique()

# TODO -> DA GESTIRE I DATI DOVE SONO SUDDIVISI IN PIÙ STATI

array(['Spagna', 'Italia', 'Malta', 'Tunisia', '82 in Italia, 2 a Malta',
       '182 in Italia, 36 a Malta', 'Francia', 'Italia (1 a Malta)', nan,
       'median # migrants rescued per mission', 70, 148, 186, 69],
      dtype=object)

In [7]:
ispi_standoffs['where disembarked (Italy)'].unique()

array([nan, 'Pozzallo', 'Catania', 'Lampedusa', 'Augusta', 'Messina',
       'Taranto', 'Palermo', 'Porto Empedocle', 'Olbia', 'Trapani',
       'Pozallo', 'Salerno', 'Livorno', 'Reggio Calabria', 'Tolone',
       'Bari', 'Gioia Tauro', 'Ravenna', 'Ancona', 'La Spezia',
       'Marina di Carrara', 'Napoli', 'Civitavecchia', 'Ortona',
       'Brindisi', 'Vibo Valentia', 'Marina di Carrara + Livorno',
       'Genova', 'Crotone'], dtype=object)

In [8]:
!pip install geopy


Defaulting to user installation because normal site-packages is not writeable


In [9]:
cols = ['city']
names = [('Roma'),('Trento'),('Genova'),('Trieste'),('Venezia')]
cities = gpd.GeoDataFrame(names,columns=cols)
geo_cities = gpd.tools.geocode(cities.city, provider="arcgis")

In [10]:
cities_ispi = ispi_standoffs[ispi_standoffs['where disembarked (Italy)'].notna()]
cities_ispi

,crisis number,ship,persons onboard,start date,end date,length of standoff,relocated (agreed),disembarked,where disembarked (Italy),nautical miles from Lampedusa
1,2,Diciotti,523,15/06/2018,19/06/2018,5,0,Italia,Pozzallo,130.0
3,4,Alexander Maersk,113,22/06/2018,26/06/2018,5,0,Italia,Pozzallo,130.0
4,5,Monte Sperone + Protector,447,13/07/2018,16/07/2018,4,270,Italia,Pozzallo,130.0
6,7,Diciotti,177,14/08/2018,25/08/2018,12,50,Italia,Catania,210.0
12,13,Sea-Watch 3,47,19/01/2019,31/01/2019,13,32,Italia,Catania,210.0
...,...,...,...,...,...,...,...,...,...,...
279,280,Nadir,49,20/11/2023,20/11/2023,1,0,Italia,Lampedusa,0.0
280,281,Aurora SAR,45,22/11/2023,22/11/2023,1,0,Italia,Lampedusa,0.0
281,282,Humanity 1,200,30/11/2023,2023-02-12 00:00:00,3,0,Italia,Crotone,330.0
282,283,Geo Barents,44,30/11/2023,2023-03-12 00:00:00,4,0,Italia,Taranto,430.0


In [11]:
geo_cities_ispi = gpd.tools.geocode(cities_ispi['where disembarked (Italy)'], provider="arcgis")

In [12]:
geo_cities_ispi

,geometry,address
1,POINT (14.8457 36.72605),"Pozzallo, Ragusa"
3,POINT (14.8457 36.72605),"Pozzallo, Ragusa"
4,POINT (14.8457 36.72605),"Pozzallo, Ragusa"
6,POINT (15.08783 37.50248),Catania
12,POINT (15.08783 37.50248),Catania
...,...,...
279,POINT (12.60982 35.50112),"Lampedusa, Lampedusa e Linosa, Agrigento"
280,POINT (12.60982 35.50112),"Lampedusa, Lampedusa e Linosa, Agrigento"
281,POINT (17.12715 39.08067),Crotone
282,POINT (17.24006 40.46922),Taranto


In [13]:
geo_cities_ispi.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [14]:
# ispi_standoff_gpd = ispi_standoffs.merge(geo_cities_ispi)
# ispi_standoff_gpd
ispi_standoff_gpd = gpd.GeoDataFrame(cities_ispi, geometry=geo_cities_ispi.geometry, crs=geo_cities_ispi.crs)

In [15]:
ispi_standoff_gpd

,crisis number,ship,persons onboard,start date,end date,length of standoff,relocated (agreed),disembarked,where disembarked (Italy),nautical miles from Lampedusa,geometry
1,2,Diciotti,523,15/06/2018,19/06/2018,5,0,Italia,Pozzallo,130.0,POINT (14.8457 36.72605)
3,4,Alexander Maersk,113,22/06/2018,26/06/2018,5,0,Italia,Pozzallo,130.0,POINT (14.8457 36.72605)
4,5,Monte Sperone + Protector,447,13/07/2018,16/07/2018,4,270,Italia,Pozzallo,130.0,POINT (14.8457 36.72605)
6,7,Diciotti,177,14/08/2018,25/08/2018,12,50,Italia,Catania,210.0,POINT (15.08783 37.50248)
12,13,Sea-Watch 3,47,19/01/2019,31/01/2019,13,32,Italia,Catania,210.0,POINT (15.08783 37.50248)
...,...,...,...,...,...,...,...,...,...,...,...
279,280,Nadir,49,20/11/2023,20/11/2023,1,0,Italia,Lampedusa,0.0,POINT (12.60982 35.50112)
280,281,Aurora SAR,45,22/11/2023,22/11/2023,1,0,Italia,Lampedusa,0.0,POINT (12.60982 35.50112)
281,282,Humanity 1,200,30/11/2023,2023-02-12 00:00:00,3,0,Italia,Crotone,330.0,POINT (17.12715 39.08067)
282,283,Geo Barents,44,30/11/2023,2023-03-12 00:00:00,4,0,Italia,Taranto,430.0,POINT (17.24006 40.46922)


In [16]:
ispi_map = leafmap.Map()

In [17]:
ispi_standoff_gpd = ispi_standoff_gpd.rename(columns={'persons onboard': 'persons_onboard'})

In [18]:
ispi_standoff_gpd.head()['disembarked']

1     Italia
3     Italia
4     Italia
6     Italia
12    Italia
Name: disembarked, dtype: object

In [19]:
'disembarked' in ispi_standoff_gpd.columns


True

In [20]:
ispi_map.add_gdf(ispi_standoff_gpd, point_style={
        "radius": 6,
        "color": "black",
        "fillColor": "red",
        "fillOpacity": 0.7,
    })
ispi_map

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [ ]:
ispi_standoff_gpd['where disembarked (Italy)'].unique()
# TODO -> verificare che la nazione di sbarco sia l'italia, verificare i nomi in modo univoco
# TODO -> SOSTITUIRE POZALLO CON POZZALLO
# todo -> specificare per tutte le voci, il fatto che sia in Italia (E.g. Augusta in U.S.A. KO)



array(['Pozzallo', 'Catania', 'Lampedusa', 'Augusta', 'Messina',
       'Taranto', 'Palermo', 'Porto Empedocle', 'Olbia', 'Trapani',
       'Pozallo', 'Salerno', 'Livorno', 'Reggio Calabria', 'Tolone',
       'Bari', 'Gioia Tauro', 'Ravenna', 'Ancona', 'La Spezia',
       'Marina di Carrara', 'Napoli', 'Civitavecchia', 'Ortona',
       'Brindisi', 'Vibo Valentia', 'Marina di Carrara + Livorno',
       'Genova', 'Crotone'], dtype=object)

In [ ]:
'Porto di ' + ispi_standoff_gpd['where disembarked (Italy)'].unique() + ', Italy'

array(['Porto di Pozzallo, Italia', 'Porto di Catania, Italia',
       'Porto di Lampedusa, Italia', 'Porto di Augusta, Italia',
       'Porto di Messina, Italia', 'Porto di Taranto, Italia',
       'Porto di Palermo, Italia', 'Porto di Porto Empedocle, Italia',
       'Porto di Olbia, Italia', 'Porto di Trapani, Italia',
       'Porto di Pozallo, Italia', 'Porto di Salerno, Italia',
       'Porto di Livorno, Italia', 'Porto di Reggio Calabria, Italia',
       'Porto di Tolone, Italia', 'Porto di Bari, Italia',
       'Porto di Gioia Tauro, Italia', 'Porto di Ravenna, Italia',
       'Porto di Ancona, Italia', 'Porto di La Spezia, Italia',
       'Porto di Marina di Carrara, Italia', 'Porto di Napoli, Italia',
       'Porto di Civitavecchia, Italia', 'Porto di Ortona, Italia',
       'Porto di Brindisi, Italia', 'Porto di Vibo Valentia, Italia',
       'Porto di Marina di Carrara + Livorno, Italia',
       'Porto di Genova, Italia', 'Porto di Crotone, Italia'],
      dtype=object)

In [47]:
ispi_standoff_disembarked_it_list = ispi_standoff_gpd.query("disembarked == 'Italia'")['where disembarked (Italy)'].unique()

In [65]:
type(ispi_standoff_disembarked_it_list)
import numpy as np

ispi_standoff_disembarked_it_list

ispi_standoff_disembarked_it_list = np.delete(ispi_standoff_disembarked_it_list, np.where(ispi_standoff_disembarked_it_list == 'Pozallo'))
ispi_standoff_disembarked_it_list = np.delete(ispi_standoff_disembarked_it_list, np.where(ispi_standoff_disembarked_it_list == 'Marina di Carrara + Livorno'))

# 10
# ispi_standoff_disembarked_it_list[np.where(ispi_standoff_disembarked_it_list == 'Pozallo')]
#np.remove(ispi_standoff_disembarked_it_list, np.where(ispi_standoff_disembarked_it_list == 'Pozallo'))
#ispi_standoff_disembarked_it_list


In [66]:
ispi_standoff_disembarked_it_list

array(['Pozzallo', 'Catania', 'Lampedusa', 'Augusta', 'Taranto',
       'Messina', 'Palermo', 'Porto Empedocle', 'Olbia', 'Trapani',
       'Salerno', 'Livorno', 'Reggio Calabria', 'Bari', 'Gioia Tauro',
       'Ravenna', 'Ancona', 'La Spezia', 'Marina di Carrara',
       'Civitavecchia', 'Ortona', 'Brindisi', 'Napoli', 'Vibo Valentia',
       'Genova', 'Crotone'], dtype=object)

In [72]:
import pandas as pd

cities = pd.Series(
    ispi_standoff_disembarked_it_list,
    name='input_location'
)

geo_cities_ispi = gpd.tools.geocode(
    cities + ', Italy',
    provider='arcgis'
)

In [76]:
geo_cities_ispi['input_location'] = ispi_standoff_disembarked_it_list
geo_cities_ispi

,geometry,address,input_location
0,POINT (14.8457 36.72605),"Pozzallo, Ragusa",Pozzallo
1,POINT (15.08783 37.50248),Catania,Catania
2,POINT (12.60982 35.50112),"Lampedusa, Lampedusa e Linosa, Agrigento",Lampedusa
3,POINT (15.219 37.23375),"Augusta, Siracusa",Augusta
4,POINT (17.24006 40.46922),Taranto,Taranto
5,POINT (15.55552 38.19233),Messina,Messina
6,POINT (13.36147 38.11566),Palermo,Palermo
7,POINT (13.5269 37.28652),"Porto Empedocle, Agrigento",Porto Empedocle
8,POINT (9.48685 40.92251),"Olbia, Sassari",Olbia
9,POINT (12.51463 38.01859),Trapani,Trapani


In [84]:
ispi_standoff_disembarked_it_gpd = ispi_standoff_gpd.query("disembarked == 'Italia'")
type(ispi_standoff_disembarked_it_gpd)
ispi_standoff_disembarked_it_gpd.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [85]:
ispi_standoff_disembarked_it_gpd = ispi_standoff_disembarked_it_gpd.drop(columns=['geometry'])


In [ ]:
ispi_standoff_disembarked_it_gpd_MR = ispi_standoff_disembarked_it_gpd.merge(geo_cities_ispi, left_on='where disembarked (Italy)', right_on='input_location')

# TODO NELLA MERGE RICORDARSI CHE I NOMI COMPOSTI O ERRATI RIMARRANO FUORI -> DA SISTEMARE

In [87]:
ispi_standoff_disembarked_it_gpd_MR.head()

,crisis number,ship,persons_onboard,start date,end date,length of standoff,relocated (agreed),disembarked,where disembarked (Italy),nautical miles from Lampedusa,geometry,address,input_location
0,2,Diciotti,523,15/06/2018,19/06/2018,5,0,Italia,Pozzallo,130.0,POINT (14.8457 36.72605),"Pozzallo, Ragusa",Pozzallo
1,4,Alexander Maersk,113,22/06/2018,26/06/2018,5,0,Italia,Pozzallo,130.0,POINT (14.8457 36.72605),"Pozzallo, Ragusa",Pozzallo
2,5,Monte Sperone + Protector,447,13/07/2018,16/07/2018,4,270,Italia,Pozzallo,130.0,POINT (14.8457 36.72605),"Pozzallo, Ragusa",Pozzallo
3,7,Diciotti,177,14/08/2018,25/08/2018,12,50,Italia,Catania,210.0,POINT (15.08783 37.50248),Catania,Catania
4,13,Sea-Watch 3,47,19/01/2019,31/01/2019,13,32,Italia,Catania,210.0,POINT (15.08783 37.50248),Catania,Catania


In [88]:
ispi_standoff_disembarked_it_gpd_MR['end date'] = pd.to_datetime(ispi_standoff_disembarked_it_gpd_MR['end date'])

In [89]:
ispi_standoff_disembarked_it_gpd_MR

,crisis number,ship,persons_onboard,start date,end date,length of standoff,relocated (agreed),disembarked,where disembarked (Italy),nautical miles from Lampedusa,geometry,address,input_location
0,2,Diciotti,523,15/06/2018,2018-06-19,5,0,Italia,Pozzallo,130.0,POINT (14.8457 36.72605),"Pozzallo, Ragusa",Pozzallo
1,4,Alexander Maersk,113,22/06/2018,2018-06-26,5,0,Italia,Pozzallo,130.0,POINT (14.8457 36.72605),"Pozzallo, Ragusa",Pozzallo
2,5,Monte Sperone + Protector,447,13/07/2018,2018-07-16,4,270,Italia,Pozzallo,130.0,POINT (14.8457 36.72605),"Pozzallo, Ragusa",Pozzallo
3,7,Diciotti,177,14/08/2018,2018-08-25,12,50,Italia,Catania,210.0,POINT (15.08783 37.50248),Catania,Catania
4,13,Sea-Watch 3,47,19/01/2019,2019-01-31,13,32,Italia,Catania,210.0,POINT (15.08783 37.50248),Catania,Catania
...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,280,Nadir,49,20/11/2023,2023-11-20,1,0,Italia,Lampedusa,0.0,POINT (12.60982 35.50112),"Lampedusa, Lampedusa e Linosa, Agrigento",Lampedusa
256,281,Aurora SAR,45,22/11/2023,2023-11-22,1,0,Italia,Lampedusa,0.0,POINT (12.60982 35.50112),"Lampedusa, Lampedusa e Linosa, Agrigento",Lampedusa
257,282,Humanity 1,200,30/11/2023,2023-02-12,3,0,Italia,Crotone,330.0,POINT (17.12715 39.08067),Crotone,Crotone
258,283,Geo Barents,44,30/11/2023,2023-03-12,4,0,Italia,Taranto,430.0,POINT (17.24006 40.46922),Taranto,Taranto


In [91]:
m_ispi_disembarked_italy_endtime = leafmap.Map()

m_ispi_disembarked_italy_endtime.add_gdf_time_slider(
    ispi_standoff_disembarked_it_gpd_MR,
    time_col='end date',
    layer_name='Events' # TODO IL LAYER È IMPORTANTE PER QUANTO RIGUARDA LA GESTIONE INIZIO/FINE STANDOFF
)

m_ispi_disembarked_italy_endtime


ValueError: No time columns found in the GeoDataFrame

In [92]:
ispi_standoff_disembarked_it_gpd_MR['end date'] = pd.to_datetime(
    ispi_standoff_disembarked_it_gpd_MR['end date'],
    errors='coerce'
)


In [93]:
ispi_standoff_disembarked_it_gpd_MR['end date'].dtype


dtype('<M8[ns]')

In [94]:
ispi_standoff_disembarked_it_gpd_MR = (
    ispi_standoff_disembarked_it_gpd_MR
    .rename(columns={'end date': 'end_date'})
)


In [98]:
ispi_standoff_disembarked_it_gpd_MR.columns

Index(['crisis number', 'ship', 'persons_onboard', 'start date', 'end_date',
       'length of standoff', 'relocated (agreed)', 'disembarked',
       'where disembarked (Italy)', 'nautical miles from Lampedusa',
       'geometry', 'address', 'input_location'],
      dtype='object')

In [115]:
import geopandas as gpd
import pandas as pd
from shapely import wkt

def build_wide_temporal_gdf(
    gdf,
    date_col='end_date',
    value_col='persons_onboard',
    location_col='input_location'
):
    gdf = gdf.copy()
    gdf[date_col] = pd.to_datetime(gdf[date_col])

    # geometry -> WKT
    gdf['_geom_wkt'] = gdf.geometry.apply(lambda g: g.wkt)

    # mapping geometry -> input_location
    loc_map = (
        gdf[['_geom_wkt', location_col]]
        .drop_duplicates()
        .groupby('_geom_wkt')[location_col]
        .agg(lambda x: x.iloc[0])
    )

    # pivot temporale
    wide = gdf.pivot_table(
        index='_geom_wkt',
        columns=date_col,
        values=value_col,
        aggfunc='sum'
    )

    # aggiunta input_location
    wide[location_col] = loc_map
    wide = wide.reset_index()

    # Timestamp -> YYYY-MM-DD
    wide.columns = [
        c.strftime('%Y-%m-%d') if isinstance(c, pd.Timestamp) else c
        for c in wide.columns
    ]

    # WKT -> geometry
    wide['geometry'] = wide['_geom_wkt'].apply(wkt.loads)
    wide = wide.drop(columns='_geom_wkt')

    # ordine colonne
    date_cols = [c for c in wide.columns if c not in ['geometry', location_col]]
    wide = wide[['geometry', location_col] + date_cols]

    return gpd.GeoDataFrame(wide, geometry='geometry')


In [116]:
gdf_wide = build_wide_temporal_gdf(
    ispi_standoff_disembarked_it_gpd_MR,
    date_col='end_date',
    value_col='persons_onboard',
    location_col='input_location'
)


In [117]:
gdf_wide

,geometry,input_location,2018-06-19,2018-06-26,2018-07-16,2018-08-25,2019-01-31,2019-02-09,2019-03-11,2019-03-12,...,2023-11-10,2023-11-11,2023-11-14,2023-11-15,2023-11-20,2023-11-22,2023-11-23,2023-11-30,2023-12-01,2023-12-08
0,POINT (10.04142 44.03837),Marina di Carrara,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,21,NaN,NaN,NaN
1,POINT (10.30784 43.55189),Livorno,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,POINT (11.79681 42.09118),Civitavecchia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,162,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,POINT (12.19659 44.41574),Ravenna,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,57,NaN,NaN,NaN,NaN
4,POINT (12.51463 38.01859),Trapani,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,POINT (12.60982 35.50112),Lampedusa,NaN,NaN,NaN,NaN,NaN,106,NaN,NaN,...,NaN,NaN,NaN,39,49,45,NaN,45,NaN,NaN
6,POINT (13.36147 38.11566),Palermo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,POINT (13.51599 43.61715),Ancona,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73,NaN
8,POINT (13.5269 37.28652),Porto Empedocle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60
9,POINT (14.25254 40.83998),Napoli,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
m_ispi_datetime = leafmap.Map()
m_ispi_datetime.add_gdf_time_slider(gdf_wide, time_interval=0.05, zoom_to_layer=True)


In [119]:
m_ispi_datetime

Map(center=[np.float64(39.958426), np.float64(13.4364899)], controls=(ZoomControl(options=['position', 'zoom_i…